In [1]:
import pandas as pd
from datetime import datetime
import app as app
import _connection_ as con
import _utils_ as utils

In [2]:
# app.lambda_handler(None, None)

### Get list of the largest stations

In [ ]:
table_name = "gbfs_station_info"
capacity_threshold = 26
disabled_docks_threshold = 1
max_num_stations = 20

# Create the dynamic query for bulk insert
query = f"""
SELECT * FROM  {table_name} 
WHERE station_id IN (
	SELECT si.station_id FROM {table_name} si
	JOIN gbfs_station_status ss ON ss.station_id = si.station_id
	WHERE capacity>{capacity_threshold}
	AND num_docks_disabled< {disabled_docks_threshold}
)
ORDER BY is_charging_station DESC LIMIT {max_num_stations}
"""
conn, cur = con.connect_to_db()
print(query)
cur.execute(query)
res  = cur.fetchall()
conn.close()
res_df = pd.DataFrame(res, columns=utils.get_table_columns(table_name))
date_now = datetime.now().strftime("%Y-%m-%d")
res_df
# stations_file_name = f"selected_stations {date_now}.csv"
# res_df.to_csv(f"selected_stations/{stations_file_name}")


In [7]:
stations_df= pd.read_csv(f"selected_stations/{stations_file_name}")

In [9]:
stations_df['station_id'][0], type(stations_df['station_id'][0])

(25, numpy.int64)

In [27]:
# DELETE DATA FROM OTHER STATIONS
sql_delete = f"""
    DELETE FROM gbfs_station_status
    WHERE station_id NOT IN (
        SELECT si.station_id FROM {table_name} si
        JOIN gbfs_station_status ss ON ss.station_id = si.station_id
        WHERE capacity>{capacity_threshold}
        AND num_docks_disabled< {disabled_docks_threshold}
    )
"""
conn, cur = con.connect_to_db()
cur.execute(sql_delete)
conn.commit()
conn.close()
